In [13]:
import pandas as pd



df = pd.read_csv("/Users/kartik/Desktop/Local Projects/Machiene learning/STOCK PRICE PREDICTION UNSUPERVISED LEARNING/DATA/stocks_100.csv")

print(df.head())
print(df.shape)

         Date        AAPL        ABBV         ABT        ADBE        AMD  \
0  2023-01-03  122.982727  143.135178  102.208015  336.920013  64.019997   
1  2023-01-04  124.251183  144.289902  103.728348  341.410004  64.660004   
2  2023-01-05  122.933563  144.113586  103.345924  328.440002  62.330002   
3  2023-01-06  127.456787  146.810959  104.773010  332.750000  63.959999   
4  2023-01-09  127.977959  142.500488  104.605103  341.980011  67.239998   

         AMGN       AMZN  AXISBANK.NS         AXP  ...         TGT  \
0  236.093933  85.820000   959.757385  140.973801  ...  134.066101   
1  238.566315  85.139999   954.920227  144.250946  ...  134.940872   
2  240.795044  83.120003   947.041138  140.798965  ...  136.301559   
3  248.320450  86.080002   937.416687  144.395142  ...  141.505829   
4  243.736649  87.360001   956.216797  144.616318  ...  138.148254   

          TMO        TSLA       UBER         UNH         VLO        WFC  \
0  547.726196  108.099998  25.360001  487.19229

In [17]:
# Fresh start — use all stocks
data = df.copy()
data['Date'] = pd.to_datetime(data['Date'])
data = data.set_index('Date')

print(data.shape)
print(data.head())


train_size = int(len(data) * 0.8)
train = data[:train_size]
test = data[train_size:]

print("Training days:", len(train))
print("Testing days:", len(test))

(237, 90)
                  AAPL        ABBV         ABT        ADBE        AMD  \
Date                                                                    
2023-01-03  122.982727  143.135178  102.208015  336.920013  64.019997   
2023-01-04  124.251183  144.289902  103.728348  341.410004  64.660004   
2023-01-05  122.933563  144.113586  103.345924  328.440002  62.330002   
2023-01-06  127.456787  146.810959  104.773010  332.750000  63.959999   
2023-01-09  127.977959  142.500488  104.605103  341.980011  67.239998   

                  AMGN       AMZN  AXISBANK.NS         AXP        BAC  ...  \
Date                                                                   ...   
2023-01-03  236.093933  85.820000   959.757385  140.973801  30.800152  ...   
2023-01-04  238.566315  85.139999   954.920227  144.250946  31.379202  ...   
2023-01-05  240.795044  83.120003   947.041138  140.798965  31.314867  ...   
2023-01-06  248.320450  86.080002   937.416687  144.395142  31.627367  ...   
2023-01-09

In [22]:
print(data.shape)  # how many columns after set_index?

(237, 90)


In [18]:
import numpy as np 
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

def create_sequences(data, window=10):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])      
        y.append(data[i+window])         
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled)
X_test, y_test = create_sequences(test_scaled)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (179, 10, 90)
X_test shape: (38, 10, 90)


In [23]:
import torch
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).to(device)

class StockLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=90, hidden_size=64, num_layers=2, batch_first=True)
        self.fc = nn.Linear(64, 90)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = StockLSTM().to(device)
print(model)

StockLSTM(
  (lstm): LSTM(90, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=90, bias=True)
)


In [24]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
epochs = 100

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    y_pred = model(X_train_t)
    loss = loss_fn(y_pred.squeeze(), y_train_t.squeeze())
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

Epoch [10/100], Loss: 0.164735
Epoch [20/100], Loss: 0.070295
Epoch [30/100], Loss: 0.053369
Epoch [40/100], Loss: 0.042920
Epoch [50/100], Loss: 0.036683
Epoch [60/100], Loss: 0.030322
Epoch [70/100], Loss: 0.025182
Epoch [80/100], Loss: 0.020561
Epoch [90/100], Loss: 0.017042
Epoch [100/100], Loss: 0.014309


In [1]:
import matplotlib.pyplot as plt

# Get predictions and actual for all stocks
model.eval()
with torch.no_grad():
    predictions = model(X_test_t).squeeze().cpu().numpy()
    actual = y_test_t.squeeze().cpu().numpy()

# Reverse scaling
predictions = scaler.inverse_transform(predictions)
actual = scaler.inverse_transform(actual)

# Pick 5 stocks to plot
stocks = list(data.columns)
plot_stocks = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
indices = [stocks.index(s) for s in plot_stocks]

# Plot 5 subplots
fig, axes = plt.subplots(5, 1, figsize=(12, 20))

for i, (ax, stock, idx) in enumerate(zip(axes, plot_stocks, indices)):
    ax.plot(actual[:, idx], label='Actual', color='blue')
    ax.plot(predictions[:, idx], label='Predicted', color='red')
    ax.set_title(f'{stock} — Actual vs Predicted')
    ax.legend()

plt.tight_layout()
plt.show()

NameError: name 'model' is not defined